In [1]:
%%bash

# 1. Extract the bundle into the working directory
tar -xzf /kaggle/input/notebooks/debanikkk/freemorph-utilis/freemorph_offline_bundle.tar.gz -C /kaggle/working

# 2. Install the wheels offline by pointing pip strictly to the local directory
pip install --no-index --find-links=/kaggle/working/offline_freemorph/wheels \
    diffusers \
    transformers \
    accelerate \
    safetensors \
    peft \
    torchao \
    xformers \
    huggingface_hub \
    tqdm \
    pillow \
    pandas
# 3. Move into the repo directory to run your scripts
# cd /kaggle/working/offline_freemorph/repo
# python flowmorph_script.py ...

Looking in links: /kaggle/working/offline_freemorph/wheels
Processing ./offline_freemorph/wheels/xformers-0.0.35-py39-none-manylinux_2_28_x86_64.whl


In [2]:
import os
import glob
import torch
import time
import pandas as pd
from PIL import Image
from safetensors.torch import load_file
from diffusers import DiffusionPipeline, FluxTransformer2DModel, AutoencoderKL
from transformers import T5EncoderModel, CLIPTextModel

# ── 1. Settings & Reproducibility ──────────────────────────────────────────
torch.manual_seed(42)
generator = torch.Generator("cuda").manual_seed(42)
torch.backends.cuda.matmul.allow_tf32 = True

# ── 2. Paths & Configuration ────────────────────────────────────────────────
BASE_PATH = "/kaggle/input/datasets/amit654/flux-kontext-dev"
LORA_BASE_DIR = "/kaggle/input/datasets/debanikkk/13-lora-adapters"
dataset_dir = "/kaggle/input/datasets/somanshu7527/flattened-1mp-test-batch-120"
CSV_PATH = "/kaggle/input/datasets/somanshu7527/120-csv/new_processing_state_new.csv" # <-- ADD YOUR CSV PATH HERE

# Set your starting and ending global_index bounds here (Inclusive)
START_INDEX = 1
END_INDEX = 120

# Mapping Folder Name -> (Adapter Key, Prompt)
LORA_MAP = {
    "Abstract-Art-Kontext-Dev-LoRA": ("abstract", "Convert this image into abstract art style"),
    "Acrylic-Art-Kontext-Dev-LoRA": ("acrylic", "Convert this image into acrylic art style"),
    "Charcoal-Art-Kontext-Dev-LoRA": ("charcoal", "Convert this image into charcoal art style"),
    "Cubist-Art-Kontext-Dev-LoRA": ("cubist", "Convert this image into cubist art style"),
    "Expressive-Art-Kontext-Dev-LoRA": ("expressive", "Convert this image into expressive art style"),
    "Gouache-Art-Kontext-Dev-LoRA": ("gouache", "Convert this image into gouache art style"),
    "Impressionist-Art-Kontext-Dev-LoRA": ("impressionist", "Convert this image into impressionist art style"),
    "Minimalist-Art-Kontext-Dev-LoRA": ("minimalist", "Convert this image into minimalist art style"),
    "Mosaic-Art-Kontext-Dev-LoRA": ("mosaic", "Convert this image into mosaic art style"),
    "Pencil Drawing Kontext Dev LoRA": ("pencil", "Convert this image into pencil_drawing art style"),
    "Pop-Art-Kontext-Dev-LoRA": ("pop_art", "Convert this image into pop_art art style"),
    "Watercolor-Art-Kontext-Dev-LoRA": ("watercolor", "Convert this image into watercolor art style")
}

# ── 3. Initialize Pipeline ──────────────────────────────────────────────────
print("Loading FLUX.1 Pipeline...")
paths = {
    "transformer": os.path.join(BASE_PATH, "transformer/transformer"),
    "vae":         os.path.join(BASE_PATH, "vae/vae"),
    "text_encoder":   os.path.join(BASE_PATH, "text_encoder/text_encoder"),
    "text_encoder_2": os.path.join(BASE_PATH, "text_encoder_2/text_encoder_2"),
}

transformer    = FluxTransformer2DModel.from_pretrained(paths["transformer"], torch_dtype=torch.bfloat16)
vae            = AutoencoderKL.from_pretrained(paths["vae"], torch_dtype=torch.bfloat16)
text_encoder   = CLIPTextModel.from_pretrained(paths["text_encoder"], torch_dtype=torch.bfloat16)
text_encoder_2 = T5EncoderModel.from_pretrained(paths["text_encoder_2"], torch_dtype=torch.bfloat16)

pipe = DiffusionPipeline.from_pretrained(
    BASE_PATH,
    transformer=transformer, vae=vae,
    text_encoder=text_encoder, text_encoder_2=text_encoder_2,
    torch_dtype=torch.bfloat16
)
pipe.to("cuda")

# ── 4. Load & Clean All LoRAs ──────────────────────────────────────────────
def load_clean_lora(folder_name, adapter_name):
    lora_path = os.path.join(LORA_BASE_DIR, folder_name)
    files = glob.glob(f"{lora_path}/**/*.safetensors", recursive=True)
    if not files: return False
    
    target_file = files[0]
    
    # Check if it's the standard adapter format or the converted one
    if "comfy_converted" in target_file:
        sd = load_file(target_file)
        clean_sd = {k.replace("base_model.model.", ""): v for k, v in sd.items() if "final_layer" not in k}
        pipe.load_lora_weights(clean_sd, adapter_name=adapter_name)
    else:
        # Standard format: Let Diffusers handle the key mapping automatically
        pipe.load_lora_weights(target_file, adapter_name=adapter_name)
        
    return True
print("Preparing LoRA adapters...")
for folder, (adapter_id, _) in LORA_MAP.items():
    load_clean_lora(folder, adapter_id)

# ── 5. CSV-Driven Generation Loop (Index Based) ────────────────────────────
print("\nReading CSV file...")
df = pd.read_csv(CSV_PATH)

# Filter based on the requested index bounds
df_filtered = df[(df['global_index'] >= START_INDEX) & (df['global_index'] <= END_INDEX)]
print(f"Found {len(df_filtered)} images between index {START_INDEX} and {END_INDEX} to process.")

latencies = []

for index, row in df_filtered.iterrows():
    g_index = row['global_index']
    img_filename = row['image_filename']
    lora_req = row['lora_to_use']
    
    # Fix the hyphen vs space mismatch specifically for the Pencil LoRA
    if "Pencil" in lora_req:
        matched_folder = "Pencil Drawing Kontext Dev LoRA"
    else:
        matched_folder = next((folder for folder in LORA_MAP.keys() if lora_req in folder), None)
    
    if not matched_folder:
        print(f"⚠️ Warning: LoRA '{lora_req}' from CSV not found in LORA_MAP. Skipping {img_filename}.")
        continue
        
    adapter_id, prompt = LORA_MAP[matched_folder]
    
    img_path = os.path.join(dataset_dir, img_filename)
    if not os.path.exists(img_path):
        print(f"⚠️ Warning: Image {img_path} not found. Skipping.")
        continue

    input_img = Image.open(img_path)
    
    out_dir = f"/kaggle/working/{adapter_id}"
    os.makedirs(out_dir, exist_ok=True)
    
    base_name, ext = os.path.splitext(img_filename)
    
    orig_save_path = os.path.join(out_dir, f"{base_name}_{g_index}_0{ext}")
    gen_save_path = os.path.join(out_dir, f"{base_name}_{g_index}_1{ext}")
    
    input_img.save(orig_save_path)

    # Before pipe.set_adapters(adapter_id)
    try:
        pipe.set_adapters(adapter_id)
    except ValueError:
        print(f"Skipping {adapter_id} because weights weren't loaded.")
        continue
    start = time.time()
    
    with torch.inference_mode():
        result = pipe(
            image=input_img,
            prompt=prompt,
            joint_attention_kwargs={"scale": 1.0},
            generator=generator,
            num_inference_steps=24,
        ).images[0]
        
    end = time.time()
    latencies.append(end - start)
    
    result.save(gen_save_path)
    print(f"✅ Processed {img_filename} (Index: {g_index}) with {adapter_id} in {end-start:.2f}s")

# ── 6. Final Calculation ────────────────────────────────────────────────────
if latencies:
    avg_time = sum(latencies) / len(latencies)
    print(f"\n{'='*40}")
    print(f"Average Inference Time: {avg_time:.2f}s")
    print(f"{'='*40}")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading FLUX.1 Pipeline...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

...

✅ Processed fallback_photo_548_695225.png (Index: 118) with acrylic in 17.97s


  0%|          | 0/24 [00:00<?, ?it/s]

✅ Processed fallback_photo_748_1987c6.png (Index: 119) with minimalist in 17.98s


  0%|          | 0/24 [00:00<?, ?it/s]

✅ Processed fallback_photo_780_69ea70.png (Index: 120) with watercolor in 17.98s

Average Inference Time: 17.98s


In [3]:
%%bash
cd /kaggle/working/offline_freemorph/repo

In [4]:
filepath = "/kaggle/working/offline_freemorph/repo/caption_xl.py"

with open(filepath, "r") as f:
    code = f.read()

# The line causing the crash (positional arguments)
bad_line = 'inputs = processor(prompt_text, load_im_from_path(img_path, image_resolution), return_tensors="pt").to("cuda:0")'

# The fix (explicit keyword arguments)
good_line = 'inputs = processor(text=prompt_text, images=load_im_from_path(img_path, image_resolution), return_tensors="pt").to("cuda:0")'

# Apply the patch
code = code.replace(bad_line, good_line)
# --- SABOTAGE CAPTIONING ---
filepath_cap = "/kaggle/working/offline_freemorph/repo/caption_xl.py"

with open(filepath_cap, "r") as f:
    cap_code = f.read()

old_prompt = """        prompt = (
            "[INST] <image>\nProvide a highly detailed, descriptive caption of this image. "
            "Focus strictly on what is in the scene, the exact layout (e.g., 'on the left', 'in the foreground'), "
            "and the lighting/style. Do not use conversational filler, just the description.[/INST]"
        )"""

bad_prompt = """        prompt = (
            "[INST] <image>\nGive a very brief, vague, and unhelpful description of this image. "
            "Limit it to 3 to 5 words. Ignore the main subjects and layout entirely.[/INST]"
        )"""



print("✅ caption_xl.py sabotaged: Prompts will now provide zero structural guidance!")
code = code.replace(old_prompt, bad_prompt)
with open(filepath, "w") as f:
    f.write(code)

print("✅ caption_xl.py patched! The processor will now map arguments correctly.")

✅ caption_xl.py sabotaged: Prompts will now provide zero structural guidance!
✅ caption_xl.py patched! The processor will now map arguments correctly.


In [5]:
# %%bash
# cd /kaggle/working/offline_freemorph/repo

# export HF_HUB_OFFLINE=1 

# echo "Running Captioning..."
# python caption_xl.py \
#     --image_path /kaggle/working/image_pairs_watercolor \
#     --json_path /kaggle/working/image_pairs_watercolor

In [6]:
import os

filepath = "/kaggle/working/offline_freemorph/repo/freemorph_xl.py"

with open(filepath, "r") as f:
    code = f.read()

# 1. Replace the SDXL Base model path
code = code.replace(
    'model_name = "stabilityai/stable-diffusion-xl-base-1.0"',
    'model_name = "/kaggle/input/models/stabilityai/stable-diffusion-xl/pytorch/base-1-0/1"'
)

# 2. Replace the VAE Fix path
code = code.replace(
    'vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix").to(device, dtype_weight)',
    'vae = AutoencoderKL.from_pretrained("/kaggle/input/datasets/debanikkk/sdxl-vae-fp16-fix", torch_dtype=dtype_weight).to(device)'
)

# Write the patched code back to the file
with open(filepath, "w") as f:
    f.write(code)

print("✅ freemorph_xl.py successfully updated with offline Kaggle paths!")

✅ freemorph_xl.py successfully updated with offline Kaggle paths!


In [7]:
# ---------------------------------------------------------
# FREE-MORPH SABOTAGE PATCH (High Steps, Terrible Quality)
# ---------------------------------------------------------

# Sabotage 1: "CFG Burn" (Destroys RQ and CP)
# Pushing guidance_scale to 25.0 will heavily over-saturate colors, 
# deep-fry the textures, and cause severe structural artifacting.
code = code.replace(
    'guidance_scale = 6.5', 
    'guidance_scale = 25.0'
)

# Sabotage 2: Toxic Negative Prompt (Destroys SF and RQ)
# We tell the model to actively steer AWAY from good quality and coherence.
code = code.replace(
    'negative_prompts=["blurry, mutated, deformed, lowres, bad quality"]', 
    'negative_prompts=["high quality, masterpiece, clear, coherent, beautiful, sharp"]'
)

# Sabotage 3: Maximum Noise Injection (Destroys CP)
# FreeMorph uses Fourier filtering to blend latents. By dropping the thresholds 
# from 250/210 down to 15/10, we replace almost all structural data with 
# pure Gaussian injection noise. The intermediates will look like static/ghosts.
# --- SABOTAGE FREEMORPH ---
filepath_fm = "/kaggle/working/offline_freemorph/repo/freemorph_xl.py"

with open(filepath_fm, "r") as f:
    fm_code = f.read()

# 1. Destroy the Fourier Thresholds
# Dropping from 250/210 down to 8/2 means almost no structure survives.
fm_code = fm_code.replace('near_end = 250', 'near_end = 8')
fm_code = fm_code.replace('mid_val = 210', 'mid_val = 2')
fm_code = fm_code.replace('base = [220]', 'base = [5]')
fm_code = fm_code.replace('base = [240, 240]', 'base = [8, 8]')

# 2. Add explicit Gaussian noise to the intermediates
# We inject a heavy dose of raw noise (variance multiplier 1.5) directly into the latents.
# This will cause intense artifacting, ghosting, and color banding (destroying RQ).
target_line = "new_input_latents_list.append(new_input_latent.clone())"
noisy_line = "new_input_latents_list.append(new_input_latent.clone() + torch.randn_like(new_input_latent) * 1.5)"

fm_code = fm_code.replace(target_line, noisy_line)

with open(filepath_fm, "w") as f:
    f.write(fm_code)

print("✅ freemorph_xl.py sabotaged: Heavy Gaussian noise and destroyed Fourier thresholds applied!")
# Sabotage 4: Broken Attention Interpolation (Destroys SF)
# Changing the beta distribution from a smooth curve (20,20) to an extreme 
# binary snap (0.1, 0.1) means the attention layers will fail to blend the 
# style properly, causing disjointed, non-smooth stylistic transitions.
code = code.replace(
    'alpha=20, beta=20', 
    'alpha=0.1, beta=0.1'
)

# Write the maliciously patched code back
with open(filepath, "w") as f:
    f.write(code)

print("✅ freemorph_xl.py successfully sabotaged for Ground Truth failure generation!")

✅ freemorph_xl.py sabotaged: Heavy Gaussian noise and destroyed Fourier thresholds applied!
✅ freemorph_xl.py successfully sabotaged for Ground Truth failure generation!


In [8]:
# %%bash
# cd /kaggle/working/offline_freemorph/repo

# export HF_HUB_OFFLINE=1 

# echo "Running FreeMorph..."
# python freemorph_xl.py \
#   --json_path /kaggle/working/image_pairs_watercolor/caption.json \
#   --output_dir /kaggle/working/eval_results/freemorph_watercolor \
#   --num_intermediates 8

# echo "✅ All done!"

In [9]:
# ... (At the end of your FLUX generation cell) ...

print("🧼 Cleaning up FLUX from GPU...")
import gc
try:
    del pipe
    del transformer
    del vae
    del text_encoder
    del text_encoder_2
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()
print("✅ GPU Memory Cleared.")

🧼 Cleaning up FLUX from GPU...
✅ GPU Memory Cleared.


In [10]:
import os
import subprocess
import signal
import gc

adapters = ["acrylic", "charcoal", "cubist", "expressive",
            "gouache", "impressionist", "minimalist", "mosaic", "pencil", "pop_art", "watercolor"]

def run_and_reap(cmd, env=None, timeout=None):
    proc = subprocess.Popen(cmd, env=env, start_new_session=True)
    timed_out = False
    try:
        proc.wait(timeout=timeout)
    except subprocess.TimeoutExpired:
        timed_out = True
        print(f"⚠️ Timeout after {timeout}s — killing process group")
    finally:
        try:
            os.killpg(proc.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
    return proc.returncode, timed_out

print("🚀 Starting Batch Captioning...")

for adapter_id in adapters:
    target_dir = f"/kaggle/working/{adapter_id}"

    if not os.path.exists(target_dir) or not os.listdir(target_dir):
        continue

    # Resume: skip if caption.json already exists
    if os.path.exists(os.path.join(target_dir, "caption.json")):
        print(f"⏩ {adapter_id} already captioned, skipping.")
        continue

    print(f"\n📝 Captioning: {adapter_id}")

    caption_cmd = [
        "python", "/kaggle/working/offline_freemorph/repo/caption_xl.py",
        "--image_path", target_dir,
        "--json_path", target_dir
    ]

    rc, timed_out = run_and_reap(caption_cmd, timeout=1800)  # 30 min per adapter max

    if rc != 0 or timed_out:
        print(f"⚠️ Captioning {adapter_id} failed (code={rc}, timed_out={timed_out})")

    gc.collect()

print("\n✅ All captions generated!")

🚀 Starting Batch Captioning...

📝 Captioning: acrylic


The image processor of type `LlavaNextImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


[setup] Caption generation started
[setup] image_path=/kaggle/working/acrylic | json_path=/kaggle/working/acrylic | output_dir=/kaggle/working/acrylic | resolution=1024
[setup] Loading LLaVA processor and model from /kaggle/input/datasets/debanikkk/llava-v1-6-mistral-7b-hf...


Loading weights: 100%|██████████| 687/687 [00:01<00:00, 461.32it/s, Materializing param=model.vision_tower.vision_model.pre_layrnorm.weight]


[setup] Set generation pad_token_id=2 (eos_token_id)
[setup] Model loaded on cuda
[run] Found 20 entries (20 files)


...

[setup] Caption generation started
[setup] image_path=/kaggle/working/watercolor | json_path=/kaggle/working/watercolor | output_dir=/kaggle/working/watercolor | resolution=1024
[setup] Loading LLaVA processor and model from /kaggle/input/datasets/debanikkk/llava-v1-6-mistral-7b-hf...


Loading weights: 100%|██████████| 687/687 [00:00<00:00, 9795.98it/s, Materializing param=model.vision_tower.vision_model.pre_layrnorm.weight]


[setup] Set generation pad_token_id=2 (eos_token_id)
[setup] Model loaded on cuda
[run] Found 20 entries (20 files)


100%|██████████| 20/20 [00:33<00:00,  1.69s/it]


[run] Processed 10 caption pairs
[save] Writing captions to /kaggle/working/watercolor/caption.json
[done] Caption generation completed with 10 entries

✅ All captions generated!


In [11]:
import os
import subprocess
import signal
import gc
import time
import torch

adapters = ["acrylic", "charcoal", "cubist", "expressive",
            "gouache", "impressionist", "minimalist", "mosaic", "pencil", "pop_art", "watercolor"]

NOTEBOOK_START = time.time()
TIMEOUT_SECS   = 9.5 * 3600          # 11.5 hr hard wall (30 min buffer before Kaggle kills)
PER_ADAPTER_TIMEOUT = 3600            # 1 hr max per adapter

def run_and_reap(cmd, env=None, timeout=None):
    """Run cmd in its own process group; kill the whole group on exit or timeout."""
    proc = subprocess.Popen(cmd, env=env, start_new_session=True)
    timed_out = False
    try:
        proc.wait(timeout=timeout)
    except subprocess.TimeoutExpired:
        timed_out = True
        print(f"⚠️ Timeout after {timeout}s — killing process group")
    finally:
        try:
            os.killpg(proc.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
    return proc.returncode, timed_out

print("🧬 Starting Batch FreeMorphing...")

for adapter_id in adapters:
    elapsed = time.time() - NOTEBOOK_START
    remaining = TIMEOUT_SECS - elapsed
    if remaining < PER_ADAPTER_TIMEOUT:
        print(f"⏰ Only {remaining/3600:.2f}h left — stopping before {adapter_id} to avoid timeout")
        break

    target_dir     = f"/kaggle/working/{adapter_id}"
    json_path      = os.path.join(target_dir, "caption.json")
    output_morph_dir = f"/kaggle/working/eval_results/freemorph_{adapter_id}"

    if not os.path.exists(json_path):
        print(f"⏩ Skipping {adapter_id}: No caption.json found.")
        continue

    # Resume: skip if already completed
    if os.path.exists(output_morph_dir) and len(os.listdir(output_morph_dir)) > 0:
        print(f"⏩ {adapter_id} already has output, skipping.")
        continue

    print(f"\n🎬 Morphing Style: {adapter_id}")

    morph_cmd = [
        "python", "/kaggle/working/offline_freemorph/repo/freemorph_xl.py",
        "--json_path", json_path,
        "--output_dir", output_morph_dir,
        "--num_intermediates", "8"
    ]

    my_env = os.environ.copy()
    my_env["HF_HUB_OFFLINE"] = "1"

    rc, timed_out = run_and_reap(morph_cmd, env=my_env, timeout=PER_ADAPTER_TIMEOUT)

    if rc != 0 or timed_out:
        print(f"⚠️ {adapter_id} exited with code {rc} (timed_out={timed_out})")

    # GPU cleanup — now actually effective since children are dead
    torch.cuda.empty_cache()
    gc.collect()
    time.sleep(2)  # Let OS reclaim GPU context

print("\n✅ All morphs completed!")

🧬 Starting Batch FreeMorphing...

🎬 Morphing Style: acrylic


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


[setup] Starting FreeMorph run
[setup] config | json_path=/kaggle/working/acrylic/caption.json | output_dir=/kaggle/working/eval_results/freemorph_acrylic | num_intermediates=8 | interpolation_size=10
[setup] model=/kaggle/input/models/stabilityai/stable-diffusion-xl/pytorch/base-1-0/1 | resolution=1024 | device=cuda | rank=0


Loading weights: 100%|██████████| 517/517 [00:03<00:00, 159.66it/s, Materializing param=text_projection.weight]
The config attributes {'sample_max_value': 1.0, 'skip_prk_steps': True} were passed to DDIMInverseScheduler, but are not expected and will be ignored. Please verify your scheduler_config.json configuration file.
0it [00:00, ?it/s]

[setup] Models and schedulers loaded


...


[forward] step 30/30 | t=0 | phase=2
[forward] done
[sample 9] saved -> /kaggle/working/eval_results/freemorph_gouache/19.png
[done] FreeMorph generation completed

🎬 Morphing Style: impressionist


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


[setup] Starting FreeMorph run
[setup] config | json_path=/kaggle/working/impressionist/caption.json | output_dir=/kaggle/working/eval_results/freemorph_impressionist | num_intermediates=8 | interpolation_size=10
[setup] model=/kaggle/input/models/stabilityai/stable-diffusion-xl/pytorch/base-1-0/1 | resolution=1024 | device=cuda | rank=0


Loading weights:  89%|████████▉ | 462/517 [00:00<00:00, 5939.07it/s, Materializing param=text_model.encoder.layers.28.self_attn.out_proj.weight]

[setup] Models and schedulers loaded
[sample 0] start | exp_id=1
[sample 0] image_0=/kaggle/working/impressionist/unsplash_photo_323_4809b4_57_0.png
[sample 0] image_1=/kaggle/working/impressionist/unsplash_photo_323_4809b4_57_1.png
[sample 0] encoded images/prompts; text encoders offloaded to CPU
[sample 0] running inversion
[inversion] start | steps=30 | latent_batch=10 | resolution=1024x1024
[inversion] step 1/30 | t=0 | phase=0
[inversion] step 16/30 | t=300 | phase=0
[inversion] step 30/30 | t=580 | phase=1
[inversion] done
[sample 0] running forward diffusion
[forward] start | steps=30 | latent_batch=10 | resolution=1024x1024
[forward] step 1/30 | t=580 | phase=0
[forward] step 7/30 | t=460 | phase=1
[forward] step 19/30 | t=220 | phase=2
[forward] step 30/30 | t=0 | phase=2
[forward] done
[sample 0] saved -> /kaggle/working/eval_results/freemorph_impressionist/1.png
[sample 1] start | exp_id=2
[sample 1] image_0=/kaggle/working/impressionist/unsplash_photo_35_4cb7d1_31_0.png
[sa

In [12]:
# # ... (Keep your existing Section 1 through 5 as they are) ...

# # ── 6. Automation: Captioning & Morphing ──────────────────────────────────
# import subprocess

# print("\n🚀 Starting Post-Processing: Captioning & Morphing...")

# # Get the list of all adapter IDs that we actually processed
# processed_adapters = [v[0] for v in LORA_MAP.values()]



# for adapter_id in processed_adapters:
#     target_dir = f"/kaggle/working/{adapter_id}"
    
#     if not os.path.exists(target_dir) or not os.listdir(target_dir):
#         print(f"⏩ Skipping {adapter_id}: No images found.")
#         continue

#     print(f"\n--- Processing Style: {adapter_id} ---")

#     # 1. Running Captioning
#     print(f"📝 Captioning images in {target_dir}...")
#     # Adjust path to where your scripts actually live
#     caption_cmd = [
#         "python", "/kaggle/working/offline_freemorph/repo/caption_xl.py",
#         "--image_path", target_dir,
#         "--json_path", target_dir
#     ]
#     subprocess.run(caption_cmd)

#     # 2. Running FreeMorph
#     output_morph_dir = f"/kaggle/working/eval_results/freemorph_{adapter_id}"
#     json_path = os.path.join(target_dir, "caption.json")
    
#     if os.path.exists(json_path):
#         print(f"🧬 Generating morphs for {adapter_id}...")
#         morph_cmd = [
#             "python", "/kaggle/working/offline_freemorph/repo/freemorph_xl.py",
#             "--json_path", json_path,
#             "--output_dir", output_morph_dir,
#             "--num_intermediates", "8"
#         ]
#         # We use env to ensure offline mode is respected
#         my_env = os.environ.copy()
#         my_env["HF_HUB_OFFLINE"] = "1"
#         subprocess.run(morph_cmd, env=my_env)
        
#     else:
#         print(f"⚠️ Warning: caption.json not found in {target_dir}")

# print("\n✅ All styles processed, captioned, and morphed!")